# SeamlessM4T v2-Large LoRA Fine-tuning for Maltese ASR — Retrain v2

**This is the retrain of the April SeamlessM4T notebook with the methodological fixes identified in the supervision review.**

## Changes from the previous version

1. **Proper train/val/test split.** The 4,481-utterance train CSV is split 90/10 into train (~4,033) and val (~448); the 498-utterance test CSV is held out. The val set drives early stopping and best-checkpoint selection. Protocol: Mainzinger & Levow (2024).

2. **LR raised from 5e-6 to 1e-5.** The previous run hit early stopping after just 5 epochs (best at epoch 2, WER 0.2896) even though val loss was monotonically descending the whole time. 5e-6 is the value from Gupta et al. (2024, MRL Workshop) but their work uses encoder adapters and length adapters rather than LoRA on q/v projections, and runs on Indic languages — the LR doesn't transfer cleanly. 1e-5 is a modest increase to address the slow-convergence pattern observed in the prior run; still within an order of magnitude of Gupta et al.'s value.

3. **Epoch ceiling raised to 20, patience to 5.** Previous run was killed by early stopping while still learning.

4. **New save directory.** `Seamless_M4Tv2_LoRA_Maltese_v2_proper_split` so the original checkpoint is preserved.

5. **Zero-shot baseline added.**

## Hyperparameters (defended in thesis methodology)

- **LR = 1e-5, warmup = 100:** raised from Gupta et al. (2024) 5e-6 by one factor-of-two to address the slow-convergence pattern observed in the prior run (5 epochs to early stopping with val loss still monotonically dropping). Still in the same order of magnitude as Gupta et al., the only published peer-reviewed reference for PEFT fine-tuning of SeamlessM4T at this scale.
- **LoRA r=32, α=64, q/v targets, dropout=0.05:** held identical across all 3 models.
- **No `modules_to_save`:** SeamlessM4T v2 was pretrained on 100 ASR languages including Maltese.
- **`tgt_lang="mlt"` at both tokenisation and generation time** — Seamless defaults to English without it.
- **Per-device batch=4, grad_accum=8:** effective batch 32 (matched across all 3 models). Smaller per-device batch because v2-Large is ~1.5 B params (the speech-to-text variant; see note below) and beam-search evaluation on a T4 is memory-bound.
- **Beam search width=5 at evaluation:** matched with Whisper.
- **Patience=5 on val WER**, seed=42.

### Note on parameter count

The previous thesis text described SeamlessM4T v2-Large as "approximately 2.3 billion parameters". `SeamlessM4Tv2ForSpeechToText` only instantiates the speech encoder and text decoder — the full multimodal model (including the text-encoder and vocoder branches used for translation and TTS) is the 2.3 B figure. The actual loaded model has ~1.5 B parameters, which is what's relevant for this comparison.


## 1. Setup

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate jiwer librosa soundfile sentencepiece

In [ ]:
!pip install -q --upgrade torchao

In [ ]:
from google.colab import drive
import os
import torch
from transformers import set_seed

# Mitigate CUDA memory fragmentation on T4 — relevant during beam-search eval
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Mount Drive
drive.mount('/content/drive')

# Reproducibility — identical seed across all 3 model notebooks
set_seed(42)

# Paths
model_id = "facebook/seamless-m4t-v2-large"
model_save_dir = "/content/drive/My Drive/ASRModels/Seamless_M4Tv2_LoRA_Maltese_v2_proper_split"
train_csv_path = "/content/drive/My Drive/Thesis Project/MASRI_HEADSET_v2/train_metadata.csv"
test_csv_path  = "/content/drive/My Drive/Thesis Project/MASRI_HEADSET_v2/test_metadata.csv"

# Maltese language code in SeamlessM4T's taxonomy (ISO 639-3)
TGT_LANG = "mlt"

os.makedirs(model_save_dir, exist_ok=True)
print(f"Models will be saved to: {model_save_dir}")

Mounted at /content/drive
Models will be saved to: /content/drive/My Drive/ASRModels/Seamless_M4Tv2_LoRA_Maltese_v2_proper_split


## 2. Data preparation

In [ ]:
from datasets import load_dataset, DatasetDict, Audio
import re

# Load all CSVs. The original train CSV becomes train+val; the test CSV is held out.
raw = load_dataset("csv", data_files={
    "trainval": train_csv_path,
    "test":     test_csv_path,
})

# Fix Windows paths -> Colab paths
drive_base_path = "/content/drive/My Drive/Thesis Project/MASRI_HEADSET_v2/"

def fix_paths_bulletproof(batch):
    old_path = batch["file_path"].replace("\\", "/")
    if "/speech/" in old_path:
        relative_path = "speech/" + old_path.split("/speech/")[-1]
    else:
        relative_path = old_path.split("/")[-1]
    batch["file_path"] = os.path.join(drive_base_path, relative_path)
    return batch

raw = raw.map(fix_paths_bulletproof)

# 90/10 train/val split of the trainval portion (Mainzinger & Levow 2024 protocol).
# Seed matches the global seed so the split is deterministic across the 3 notebooks.
split = raw["trainval"].train_test_split(test_size=0.10, seed=42, shuffle=True)

masri_dataset = DatasetDict({
    "train": split["train"],
    "val":   split["test"],     # 10% carved out of trainval — used for early stopping
    "test":  raw["test"],       # held-out test set — touched only at the very end
})

print(f"Train samples: {len(masri_dataset['train'])}")
print(f"Val   samples: {len(masri_dataset['val'])}")
print(f"Test  samples: {len(masri_dataset['test'])}  (held out — touched only at zero-shot baseline and final evaluation)")

# Text normalisation
def clean_text(batch):
    text = batch["transcription"].lower()
    text = re.sub(r"[^\w\s'-]", "", text)
    batch["transcription"] = re.sub(r"\s+", " ", text).strip()
    return batch

masri_dataset = masri_dataset.map(clean_text)

# Cast audio column to 16 kHz
masri_dataset = masri_dataset.cast_column("file_path", Audio(sampling_rate=16000))


Generating trainval split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4481 [00:00<?, ? examples/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

Train samples: 4032
Val   samples: 449
Test  samples: 498  (held out — touched only at zero-shot baseline and final evaluation)


Map:   0%|          | 0/4032 [00:00<?, ? examples/s]

Map:   0%|          | 0/449 [00:00<?, ? examples/s]

Map:   0%|          | 0/498 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(model_id)
print("SeamlessM4T processor loaded.")

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/5.17M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

SeamlessM4T processor loaded.


**Critical:** `tgt_lang="mlt"` is passed to the tokenizer here. This prepends the Maltese language token to each label sequence. Without it, the tokenizer defaults to English and the model is trained to emit English tokens for Maltese audio.

In [ ]:
def prepare_dataset(batch):
    audio = batch["file_path"]

    # Audio features
    audio_inputs = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
    )
    batch["input_features"] = audio_inputs.input_features[0]

    # Text labels — tgt_lang prepends the Maltese language token
    text_inputs = processor.tokenizer(
        batch["transcription"],
        tgt_lang=TGT_LANG,
    )
    batch["labels"] = text_inputs.input_ids
    return batch

masri_dataset = masri_dataset.map(
    prepare_dataset,
    remove_columns=masri_dataset.column_names["train"],
    num_proc=8,
)

print("Preprocessing complete.")

Map (num_proc=8):   0%|          | 0/4032 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/449 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/498 [00:00<?, ? examples/s]

Preprocessing complete.


## 3. Data collator and metrics

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from jiwer import wer as compute_wer, cer as compute_cer
import numpy as np

@dataclass
class DataCollatorSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids":      f["labels"]}         for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSeq2SeqWithPadding(processor=processor)

# Same normaliser as training-time, applied to BOTH preds and refs at scoring
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s'-]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Clamp Seamless's occasional out-of-range predicted IDs to pad
    vocab_size = processor.tokenizer.vocab_size
    pred_ids = np.where(
        (pred_ids >= 0) & (pred_ids < vocab_size),
        pred_ids,
        processor.tokenizer.pad_token_id,
    )

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str  = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {"wer": 1.0, "cer": 1.0}
    pred_str, label_str = zip(*pairs)

    return {
        "wer": compute_wer(list(label_str), list(pred_str)),
        "cer": compute_cer(list(label_str), list(pred_str)),
    }

## 4. Model and LoRA configuration

In [ ]:
from transformers import SeamlessM4Tv2ForSpeechToText
from peft import LoraConfig, get_peft_model
from functools import partial

# Load base model
model = SeamlessM4Tv2ForSpeechToText.from_pretrained(model_id)

# Disable cache (incompatible with gradient_checkpointing during training)
model.config.use_cache = False

# REQUIRED with gradient_checkpointing + PEFT — gradients won't flow without this
model.enable_input_require_grads()

# LoRA config — IDENTICAL to the other 2 model notebooks for fair comparison
config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    # No modules_to_save: Seamless's decoder head is already Maltese-aware
)

model = get_peft_model(model, config)

# Bind tgt_lang="mlt" to all generate() calls so eval produces Maltese
model.generate = partial(model.generate, tgt_lang=TGT_LANG)

model.print_trainable_parameters()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1429 [00:00<?, ?it/s]

SeamlessM4Tv2ForSpeechToText LOAD REPORT from: facebook/seamless-m4t-v2-large
Key                                                                           | Status     |  | 
------------------------------------------------------------------------------+------------+--+-
t2u_model.model.decoder.duration_predictor.proj.weight                        | UNEXPECTED |  | 
vocoder.hifi_gan.resblocks.{0...14}.convs2.{0, 1, 2}.bias                     | UNEXPECTED |  | 
t2u_model.model.decoder.layers.{0, 1, 2, 3, 4, 5}.self_attn.k_proj.bias       | UNEXPECTED |  | 
t2u_model.model.decoder.layers.{0, 1, 2, 3, 4, 5}.conv_layer_norm.weight      | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn.out_proj.bias                          | UNEXPECTED |  | 
text_encoder.layers.{0...23}.self_attn_layer_norm.weight                      | UNEXPECTED |  | 
t2u_model.model.decoder.layers.{0, 1, 2, 3, 4, 5}.conv_layer_norm.bias        | UNEXPECTED |  | 
vocoder.hifi_gan.resblocks.{0...14}.convs1.{0, 1,

generation_config.json: 0.00B [00:00, ?B/s]

trainable params: 6,291,456 || all params: 1,508,133,696 || trainable%: 0.4172


## 5. Trainer setup

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir=model_save_dir,
    per_device_train_batch_size=4,            # smaller than other 2 due to model size
    per_device_eval_batch_size=2,             # beam search expands memory ~5x
    gradient_accumulation_steps=8,            # effective batch size 32 (matched across all 3 models)
    learning_rate=1e-5,                       # raised from Gupta et al. 2024's 5e-6 to address slow convergence
    warmup_steps=100,
    num_train_epochs=20,                      # raised from 10 — previous run early-stopped while still learning
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=225,
    generation_num_beams=5,                   # matched with Whisper for fair seq2seq eval
    logging_steps=50,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",              # evaluated on VAL, not test
    greater_is_better=False,
    remove_unused_columns=False,
    gradient_checkpointing=True,              # paired with enable_input_require_grads() above
    seed=42,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=masri_dataset["train"],
    eval_dataset=masri_dataset["val"],        # VAL, not test
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

## 6. Zero-shot baseline on the test set

In [ ]:
print("=== Zero-shot baseline (SeamlessM4T v2-Large, no fine-tuning) on TEST set ===")
zero_shot_metrics = trainer.evaluate(
    eval_dataset=masri_dataset["test"],
    metric_key_prefix="zero_shot",
)
for k, v in zero_shot_metrics.items():
    print(f"  {k}: {v}")

import json as _json
with open(os.path.join(model_save_dir, "zero_shot_test_metrics.json"), "w") as f:
    _json.dump({k: float(v) if isinstance(v, (int, float)) else v
                for k, v in zero_shot_metrics.items()}, f, indent=2)

=== Zero-shot baseline (SeamlessM4T v2-Large, no fine-tuning) on TEST set ===


Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  zero_shot_loss: 4.3778486251831055
  zero_shot_model_preparation_time: 0.0263
  zero_shot_wer: 0.29017429611185597
  zero_shot_cer: 0.08606849527351619
  zero_shot_runtime: 548.7824
  zero_shot_samples_per_second: 0.907
  zero_shot_steps_per_second: 0.454


## 7. Training

In [ ]:
print("Starting SeamlessM4T v2-Large LoRA fine-tuning (early stopping on val WER, patience=5)")
trainer.train()

Starting SeamlessM4T v2-Large LoRA fine-tuning (early stopping on val WER, patience=5)


Epoch,Training Loss,Validation Loss,Model Preparation Time,Wer,Cer
1,83.165967,3.935702,0.026300,0.316646,0.095705
2,74.722554,3.342298,0.026300,0.313100,0.093096
3,67.113374,2.824683,0.026300,0.307468,0.089817
4,52.833232,2.335964,0.026300,0.305173,0.085602
5,44.070405,2.064503,0.026300,0.306008,0.084699
6,34.146548,1.879345,0.026300,0.305799,0.083227
7,29.052529,1.651809,0.026300,0.308719,0.084164
8,24.247295,1.455929,0.026300,0.309971,0.083562
9,22.144358,1.335996,0.026300,0.309971,0.084432


Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TrainOutput(global_step=1134, training_loss=49.02613251970348, metrics={'train_runtime': 8642.5894, 'train_samples_per_second': 9.331, 'train_steps_per_second': 0.292, 'total_flos': 1.6363785247025971e+19, 'train_loss': 49.02613251970348, 'epoch': 9.0})

## 8. Save and final test-set evaluation

In [ ]:
# Save best model (loaded automatically thanks to load_best_model_at_end)
trainer.save_model(model_save_dir)
processor.save_pretrained(model_save_dir)
print(f"Model saved to {model_save_dir}")

# FINAL evaluation on the held-out test set
print("\n=== FINAL test-set evaluation (held-out, untouched during training) ===")
final_metrics = trainer.evaluate(
    eval_dataset=masri_dataset["test"],
    metric_key_prefix="final_test",
)
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

with open(os.path.join(model_save_dir, "final_test_metrics.json"), "w") as f:
    _json.dump({k: float(v) if isinstance(v, (int, float)) else v
                for k, v in final_metrics.items()}, f, indent=2)
print(f"\nMetrics saved to {model_save_dir}/final_test_metrics.json")

Model saved to /content/drive/My Drive/ASRModels/Seamless_M4Tv2_LoRA_Maltese_v2_proper_split

=== FINAL test-set evaluation (held-out, untouched during training) ===


Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  final_test_loss: 2.2717740535736084
  final_test_model_preparation_time: 0.0263
  final_test_wer: 0.29343037732235205
  final_test_cer: 0.09223616922361692
  final_test_runtime: 497.5608
  final_test_samples_per_second: 1.001
  final_test_steps_per_second: 0.5
  epoch: 9.0

Metrics saved to /content/drive/My Drive/ASRModels/Seamless_M4Tv2_LoRA_Maltese_v2_proper_split/final_test_metrics.json
